In [1]:
# ============================================
# CELL 1: Install Dependencies
# ============================================

!pip install -q langchain langchain-openai langchain-community chromadb pypdf tiktoken openai faiss-cpu
!pip install -q streamlit streamlit-chat pyngrok
!pip install -q google-colab

print("✅ All dependencies installed!")

# ============================================
# CELL 2: Import Libraries and Setup
# ============================================

import os
import sys
import logging
from typing import List, Dict, Tuple, Optional
from pathlib import Path
import time
import pickle

import numpy as np
from google.colab import userdata, drive
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from IPython.display import Markdown

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.schema import Document

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Libraries imported!")

# ============================================
# CELL 3: Mount Google Drive (Optional - for persistent storage)
# ============================================

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Create project directory in Drive for persistence
PROJECT_DIR = "/content/drive/MyDrive/RAG_Chatbot"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/vector_db", exist_ok=True)

print(f"✅ Project directory: {PROJECT_DIR}")

# ============================================
# CELL 4: Set OpenAI API Key
# ============================================

# Method 1: Using Colab secrets (recommended)
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("✅ API Key loaded from Colab secrets!")
except:
    # Method 2: Manual input
    OPENAI_API_KEY = input("Enter your OpenAI API Key: ")
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    print("✅ API Key set!")

# Verify API key is set
if not os.getenv('OPENAI_API_KEY'):
    raise ValueError("Please set your OpenAI API key!")

# ============================================
# CELL 5: Document Ingestion Pipeline for Colab
# ============================================

class ColabDocumentIngestionPipeline:
    """
    Document ingestion pipeline optimized for Google Colab
    """

    def __init__(
        self,
        data_dir: str = None,
        vector_store_path: str = None,
        chunk_size: int = 1000,
        chunk_overlap: int = 200,
        embedding_model: str = "text-embedding-3-small"
    ):
        # Use Drive paths if available, otherwise local
        if data_dir is None:
            data_dir = f"{PROJECT_DIR}/data" if 'PROJECT_DIR' in globals() else "/content/data"
        if vector_store_path is None:
            vector_store_path = f"{PROJECT_DIR}/vector_db" if 'PROJECT_DIR' in globals() else "/content/vector_db"

        self.data_dir = Path(data_dir)
        self.vector_store_path = Path(vector_store_path)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.embeddings = OpenAIEmbeddings(model=embedding_model)

        # Create directories
        self.data_dir.mkdir(parents=True, exist_ok=True)
        self.vector_store_path.mkdir(parents=True, exist_ok=True)

        # Text splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )

    def create_sample_documents(self):
        """Create sample documents for testing"""
        sample_docs = [
            {
                "name": "ai_intro.txt",
                "content": """
                Artificial Intelligence (AI) is the simulation of human intelligence in machines
                that are programmed to think and learn. AI can be categorized into:

                1. Narrow AI: Designed for specific tasks like facial recognition or language translation
                2. General AI: Systems with human-like intelligence across various domains
                3. Super AI: AI that surpasses human intelligence

                Key AI Technologies:
                - Machine Learning: Algorithms that learn from data
                - Deep Learning: Neural networks with multiple layers
                - Natural Language Processing: Understanding and generating human language
                - Computer Vision: Interpreting visual information

                Applications of AI:
                - Healthcare: Disease diagnosis, drug discovery
                - Finance: Fraud detection, algorithmic trading
                - Transportation: Autonomous vehicles, traffic prediction
                - Entertainment: Content recommendation, game AI
                """
            },
            {
                "name": "rag_explained.txt",
                "content": """
                Retrieval-Augmented Generation (RAG) is an AI framework that combines
                retrieval systems with large language models to generate more accurate and
                contextually relevant responses.

                How RAG Works:
                1. Query Processing: User question is processed
                2. Document Retrieval: Relevant documents are retrieved from vector database
                3. Context Augmentation: Retrieved documents are added to the prompt
                4. Generation: LLM generates response using the augmented context

                Benefits of RAG:
                - Reduced hallucinations through factual grounding
                - Access to up-to-date information without retraining
                - Transparency through source citation
                - Cost-effective compared to fine-tuning

                RAG vs Fine-tuning:
                RAG is better for knowledge-intensive tasks and when data changes frequently,
                while fine-tuning is better for style adaptation and specific formatting needs.
                """
            },
            {
                "name": "langchain_guide.txt",
                "content": """
                LangChain is a framework for developing applications powered by language models.
                It provides modular components for building LLM applications.

                Core Components:
                - Models: LLM wrappers (OpenAI, Anthropic, etc.)
                - Prompts: Template management and optimization
                - Chains: Combining multiple components
                - Agents: Decision-making systems
                - Memory: Conversation context management
                - Retrievers: Document retrieval systems

                Popular Use Cases:
                - Chatbots with memory
                - Document question answering
                - Code understanding and generation
                - Data analysis and visualization

                Integration with RAG:
                LangChain provides seamless integration with vector stores like Chroma,
                FAISS, and Pinecone, making it ideal for building RAG applications.
                """
            }
        ]

        for doc in sample_docs:
            file_path = self.data_dir / doc["name"]
            if not file_path.exists():
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(doc["content"].strip())
                print(f"✅ Created: {file_path}")
            else:
                print(f"⚠️ Already exists: {file_path}")

    def load_documents(self):
        """Load documents from directory"""
        documents = []

        # Load text files
        for txt_file in self.data_dir.glob("**/*.txt"):
            try:
                loader = TextLoader(str(txt_file), encoding="utf-8")
                docs = loader.load()
                documents.extend(docs)
                print(f"📄 Loaded: {txt_file.name}")
            except Exception as e:
                print(f"❌ Error loading {txt_file}: {e}")

        # Load PDF files if any
        for pdf_file in self.data_dir.glob("**/*.pdf"):
            try:
                loader = PyPDFLoader(str(pdf_file))
                docs = loader.load()
                documents.extend(docs)
                print(f"📄 Loaded: {pdf_file.name}")
            except Exception as e:
                print(f"❌ Error loading {pdf_file}: {e}")

        return documents

    def process_documents(self, documents):
        """Split documents into chunks"""
        if not documents:
            return []

        chunks = self.text_splitter.split_documents(documents)
        print(f"✂️ Split {len(documents)} documents into {len(chunks)} chunks")

        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i

        return chunks

    def create_vector_store(self, chunks, use_faiss=True):
        """Create vector store"""
        if not chunks:
            raise ValueError("No chunks to create vector store")

        if use_faiss:
            vector_store = FAISS.from_documents(chunks, self.embeddings)
            # Save FAISS index
            faiss_path = self.vector_store_path / "faiss_index"
            vector_store.save_local(str(faiss_path))
            print(f"✅ FAISS vector store created with {len(chunks)} chunks")
        else:
            vector_store = Chroma.from_documents(
                documents=chunks,
                embedding=self.embeddings,
                persist_directory=str(self.vector_store_path / "chroma_db")
            )
            print(f"✅ Chroma vector store created with {len(chunks)} chunks")

        return vector_store

    def load_existing_vector_store(self, use_faiss=True):
        """Load existing vector store"""
        if use_faiss:
            faiss_path = self.vector_store_path / "faiss_index"
            if faiss_path.exists():
                return FAISS.load_local(
                    str(faiss_path),
                    self.embeddings,
                    allow_dangerous_deserialization=True
                )
        else:
            chroma_path = self.vector_store_path / "chroma_db"
            if chroma_path.exists():
                return Chroma(
                    persist_directory=str(chroma_path),
                    embedding_function=self.embeddings
                )
        return None

    def run_pipeline(self, use_faiss=True):
        """Run complete ingestion pipeline"""
        # Try to load existing
        existing_store = self.load_existing_vector_store(use_faiss)
        if existing_store:
            print("📚 Using existing vector store")
            return existing_store

        # Create sample documents if none exist
        if not list(self.data_dir.glob("**/*.txt")) and not list(self.data_dir.glob("**/*.pdf")):
            print("📝 No documents found. Creating sample documents...")
            self.create_sample_documents()

        # Load and process documents
        documents = self.load_documents()
        if not documents:
            raise ValueError("No documents loaded")

        chunks = self.process_documents(documents)
        vector_store = self.create_vector_store(chunks, use_faiss)

        return vector_store

print("✅ Document ingestion pipeline ready!")

# ============================================
# CELL 6: RAG Chatbot Class
# ============================================

class ColabRAGChatbot:
    """
    RAG Chatbot optimized for Google Colab with interactive widgets
    """

    def __init__(
        self,
        model_name: str = "gpt-3.5-turbo",
        temperature: float = 0.7,
        max_tokens: int = 1000,
        use_faiss: bool = True
    ):
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.use_faiss = use_faiss

        # Initialize components
        self.vector_store = None
        self.llm = None
        self.memory = None
        self.chain = None
        self.conversation_history = []

        # Initialize everything
        self._init_vector_store()
        self._init_llm()
        self._init_memory()
        self._init_chain()

    def _init_vector_store(self):
        """Initialize vector store"""
        try:
            pipeline = ColabDocumentIngestionPipeline()
            self.vector_store = pipeline.run_pipeline(use_faiss=self.use_faiss)
            print("✅ Vector store initialized")
        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")

    def _init_llm(self):
        """Initialize language model"""
        try:
            self.llm = ChatOpenAI(
                model=self.model_name,
                temperature=self.temperature,
                max_tokens=self.max_tokens
            )
            print("✅ LLM initialized")
        except Exception as e:
            print(f"❌ Error initializing LLM: {e}")

    def _init_memory(self):
        """Initialize conversation memory"""
        self.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="answer"
        )

    def _init_chain(self):
        """Initialize conversation chain"""
        if not self.llm or not self.vector_store:
            print("⚠️ Cannot initialize chain: missing components")
            return

        custom_template = """
        You are a helpful AI assistant with access to a knowledge base.
        Use the following context to answer questions accurately.
        If you don't know the answer, say so honestly.

        Context from knowledge base:
        {context}

        Chat History:
        {chat_history}

        Current Question: {question}

        Instructions:
        1. Use the provided context when relevant
        2. Cite specific information from the context
        3. If context doesn't have the info, use your general knowledge
        4. Maintain conversation context

        Answer: """

        QA_PROMPT = PromptTemplate(
            template=custom_template,
            input_variables=["context", "chat_history", "question"]
        )

        try:
            self.chain = ConversationalRetrievalChain.from_llm(
                llm=self.llm,
                retriever=self.vector_store.as_retriever(
                    search_type="similarity",
                    search_kwargs={"k": 4}
                ),
                memory=self.memory,
                combine_docs_chain_kwargs={"prompt": QA_PROMPT},
                return_source_documents=True,
                verbose=False
            )
            print("✅ Conversation chain initialized")
        except Exception as e:
            print(f"❌ Error initializing chain: {e}")

    def get_response(self, query: str) -> Tuple[str, List]:
        """Get response for query"""
        if not self.chain:
            return "Chatbot not properly initialized. Please check configuration.", []

        try:
            result = self.chain.invoke({"question": query})
            answer = result.get("answer", "Sorry, couldn't generate response.")
            sources = result.get("source_documents", [])

            # Save to conversation history
            self.conversation_history.append({
                "user": query,
                "assistant": answer,
                "sources": sources
            })

            return answer, sources

        except Exception as e:
            return f"Error: {str(e)}", []

    def clear_memory(self):
        """Clear conversation memory"""
        self._init_memory()
        if self.chain:
            self.chain.memory = self.memory
        self.conversation_history = []
        print("✅ Conversation memory cleared")

print("✅ RAG Chatbot class ready!")

# ============================================
# CELL 7: Create Interactive Chat Interface
# ============================================

class ColabChatInterface:
    """
    Interactive chat interface for Google Colab
    """

    def __init__(self):
        self.chatbot = None
        self.setup_ui()

    def setup_ui(self):
        """Setup the interactive UI"""
        # Create output area for chat
        self.chat_output = widgets.Output()

        # Create input widgets
        self.question_input = widgets.Textarea(
            placeholder="Ask me anything about your documents...",
            description="Question:",
            layout=widgets.Layout(width='80%', height='100px')
        )

        self.send_button = widgets.Button(
            description="Send",
            button_style="primary",
            layout=widgets.Layout(width='100px')
        )
        self.send_button.on_click(self.send_message)

        self.clear_button = widgets.Button(
            description="Clear History",
            button_style="warning",
            layout=widgets.Layout(width='120px')
        )
        self.clear_button.on_click(self.clear_history)

        # Configuration widgets
        self.init_button = widgets.Button(
            description="Initialize Chatbot",
            button_style="success",
            layout=widgets.Layout(width='150px')
        )
        self.init_button.on_click(self.initialize_chatbot)

        self.model_select = widgets.Dropdown(
            options=['gpt-3.5-turbo', 'gpt-4'],
            value='gpt-3.5-turbo',
            description='Model:',
            layout=widgets.Layout(width='250px')
        )

        self.temp_slider = widgets.FloatSlider(
            value=0.7,
            min=0.0,
            max=1.0,
            step=0.1,
            description='Temperature:',
            layout=widgets.Layout(width='300px')
        )

        self.status_label = widgets.Label(value="⚠️ Not initialized")

        # Layout
        config_box = widgets.VBox([
            widgets.HTML("<h3>⚙️ Configuration</h3>"),
            self.model_select,
            self.temp_slider,
            self.init_button,
            widgets.HTML("<hr>")
        ])

        control_box = widgets.HBox([
            self.question_input,
            widgets.VBox([self.send_button, self.clear_button])
        ])

        main_box = widgets.VBox([
            widgets.HTML("<h1>🤖 RAG Chatbot for Google Colab</h1>"),
            widgets.HTML("<p>Context-Aware Chatbot with Document Retrieval</p>"),
            widgets.HTML("<hr>"),
            config_box,
            self.status_label,
            widgets.HTML("<hr>"),
            widgets.HTML("<h3>💬 Chat History</h3>"),
            self.chat_output,
            control_box
        ])

        display(main_box)

    def initialize_chatbot(self, button):
        """Initialize the chatbot"""
        with self.chat_output:
            clear_output(wait=True)
            print("🔄 Initializing chatbot... Please wait...")

        self.status_label.value = "🔄 Initializing..."

        try:
            self.chatbot = ColabRAGChatbot(
                model_name=self.model_select.value,
                temperature=self.temp_slider.value
            )
            self.status_label.value = "✅ Chatbot initialized and ready!"

            with self.chat_output:
                clear_output(wait=True)
                print("✅ Chatbot initialized successfully!")
                print("\nYou can now ask questions about your documents.")
                print("\n📚 Sample questions:")
                print("- What is Artificial Intelligence?")
                print("- Explain how RAG works")
                print("- What are the benefits of LangChain?")

        except Exception as e:
            self.status_label.value = f"❌ Initialization failed"
            with self.chat_output:
                clear_output(wait=True)
                print(f"❌ Error: {str(e)}")

    def send_message(self, button):
        """Send a message and get response"""
        if not self.chatbot:
            with self.chat_output:
                clear_output(wait=True)
                print("⚠️ Please initialize the chatbot first!")
            return

        question = self.question_input.value.strip()
        if not question:
            return

        # Clear input
        self.question_input.value = ""

        with self.chat_output:
            # Display user question
            print(f"\n👤 **You:** {question}\n")

            # Get and display response
            print("🤖 **Assistant:** ")
            response, sources = self.chatbot.get_response(question)
            print(response)

            # Display sources if available
            if sources:
                print("\n📚 **Sources:**")
                for i, doc in enumerate(sources[:3], 1):  # Show top 3 sources
                    content_preview = doc.page_content[:200] + "..."
                    print(f"  {i}. {content_preview}")
                    if doc.metadata:
                        print(f"     Metadata: {doc.metadata}")
            print("\n" + "-"*50)

    def clear_history(self, button):
        """Clear chat history"""
        if self.chatbot:
            self.chatbot.clear_memory()
            with self.chat_output:
                clear_output(wait=True)
                print("🗑️ Chat history cleared!")
                print("You can start a new conversation.")
        else:
            with self.chat_output:
                clear_output(wait=True)
                print("⚠️ Chatbot not initialized")

# Initialize the interface
print("🚀 Starting RAG Chatbot Interface...")
print("⚠️ IMPORTANT: Click 'Initialize Chatbot' to start!")
print("\n" + "="*50)

chat_interface = ColabChatInterface()

# ============================================
# CELL 8: Alternative - Simple Text Interface
# ============================================

def simple_chat_interface():
    """
    Simple text-based chat interface for quick testing
    """
    print("🤖 RAG Chatbot - Simple Interface")
    print("="*50)

    # Initialize chatbot
    print("Initializing chatbot...")
    chatbot = ColabRAGChatbot()
    print("✅ Ready! Type 'quit' to exit, 'clear' to clear history\n")

    while True:
        try:
            user_input = input("👤 You: ").strip()

            if user_input.lower() == 'quit':
                print("Goodbye! 👋")
                break
            elif user_input.lower() == 'clear':
                chatbot.clear_memory()
                print("✅ History cleared!\n")
                continue
            elif not user_input:
                continue

            print("🤖 Assistant: ", end="")
            response, sources = chatbot.get_response(user_input)
            print(response)

            if sources:
                print("\n📚 Sources:")
                for i, doc in enumerate(sources[:2], 1):
                    print(f"  {i}. {doc.page_content[:150]}...")
            print()

        except KeyboardInterrupt:
            print("\n\nGoodbye! 👋")
            break
        except Exception as e:
            print(f"Error: {e}\n")

# Uncomment to use simple interface instead of widget interface
# simple_chat_interface()

# ============================================
# CELL 9: Upload Custom Documents
# ============================================

from google.colab import files

def upload_documents():
    """
    Upload documents to use with the chatbot
    """
    print("📤 Upload Documents")
    print("="*50)
    print("Supported formats: .txt, .pdf\n")

    uploaded = files.upload()

    if uploaded:
        target_dir = f"{PROJECT_DIR}/data" if 'PROJECT_DIR' in globals() else "/content/data"
        os.makedirs(target_dir, exist_ok=True)

        for filename, content in uploaded.items():
            filepath = os.path.join(target_dir, filename)
            with open(filepath, 'wb') as f:
                f.write(content)
            print(f"✅ Saved: {filename}")

        print("\n🔄 Documents uploaded! Re-initialize the chatbot to use them.")
    else:
        print("No files uploaded.")

# Upload documents (run this cell to add your own documents)
upload_documents()

# ============================================
# CELL 10: View and Manage Documents
# ============================================

def list_documents():
    """List all documents in the data directory"""
    data_dir = f"{PROJECT_DIR}/data" if 'PROJECT_DIR' in globals() else "/content/data"

    print("📚 Documents in Knowledge Base")
    print("="*50)

    txt_files = list(Path(data_dir).glob("**/*.txt"))
    pdf_files = list(Path(data_dir).glob("**/*.pdf"))

    if txt_files or pdf_files:
        print("\n📄 Text Files:")
        for f in txt_files:
            size = f.stat().st_size / 1024
            print(f"  - {f.name} ({size:.1f} KB)")

        print("\n📑 PDF Files:")
        for f in pdf_files:
            size = f.stat().st_size / 1024
            print(f"  - {f.name} ({size:.1f} KB)")
    else:
        print("No documents found. Upload some or create sample documents!")

    print(f"\n📁 Directory: {data_dir}")

# List current documents
list_documents()

# ============================================
# CELL 11: Test the Chatbot
# ============================================

def test_chatbot():
    """
    Quick test function to verify chatbot is working
    """
    print("🧪 Testing RAG Chatbot")
    print("="*50)

    # Initialize
    print("1. Initializing chatbot...")
    chatbot = ColabRAGChatbot()

    # Test queries
    test_queries = [
        "What is Artificial Intelligence?",
        "Explain RAG in simple terms",
        "What are the benefits of using LangChain?"
    ]

    print("\n2. Running test queries...\n")

    for i, query in enumerate(test_queries, 1):
        print(f"Test {i}: {query}")
        print("-" * 40)
        response, sources = chatbot.get_response(query)
        print(f"Response: {response[:200]}...")
        print(f"Sources found: {len(sources)}")
        print()

    print("✅ Testing completed!")

# Run the test
test_chatbot()

# ============================================
# CELL 12: Save Chatbot State
# ============================================

def save_chatbot_state(chatbot, filename="chatbot_state.pkl"):
    """Save chatbot state to drive"""
    try:
        # Save conversation history and configuration
        state = {
            'conversation_history': chatbot.conversation_history,
            'model_name': chatbot.model_name,
            'temperature': chatbot.temperature,
            'timestamp': time.time()
        }

        save_path = f"{PROJECT_DIR}/{filename}"
        with open(save_path, 'wb') as f:
            pickle.dump(state, f)

        print(f"✅ Chatbot state saved to {save_path}")
    except Exception as e:
        print(f"❌ Error saving state: {e}")

def load_chatbot_state(filename="chatbot_state.pkl"):
    """Load chatbot state from drive"""
    try:
        load_path = f"{PROJECT_DIR}/{filename}"
        with open(load_path, 'rb') as f:
            state = pickle.load(f)

        print(f"✅ Loaded state from {load_path}")
        print(f"   - Model: {state['model_name']}")
        print(f"   - Messages: {len(state['conversation_history'])}")
        print(f"   - Last updated: {time.ctime(state['timestamp'])}")

        return state
    except Exception as e:
        print(f"❌ Error loading state: {e}")
        return None

# Example usage:
# save_chatbot_state(chat_interface.chatbot)
# loaded_state = load_chatbot_state()

# ============================================
# CELL 13: Export Chat History
# ============================================

def export_chat_history(chatbot, format="txt"):
    """
    Export conversation history to file
    """
    if not chatbot.conversation_history:
        print("No conversation history to export")
        return

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"chat_history_{timestamp}.{format}"
    filepath = f"{PROJECT_DIR}/{filename}"

    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write("RAG Chatbot Conversation History\n")
            f.write("="*50 + "\n\n")

            for i, conv in enumerate(chatbot.conversation_history, 1):
                f.write(f"Conversation {i}\n")
                f.write("-"*30 + "\n")
                f.write(f"User: {conv['user']}\n\n")
                f.write(f"Assistant: {conv['assistant']}\n")

                if conv.get('sources'):
                    f.write("\nSources:\n")
                    for j, doc in enumerate(conv['sources'][:2], 1):
                        f.write(f"  {j}. {doc.page_content[:150]}...\n")
                f.write("\n" + "="*50 + "\n\n")

        print(f"✅ Chat history exported to: {filepath}")
        return filepath
    except Exception as e:
        print(f"❌ Error exporting: {e}")
        return None

# Export current conversation (run after chatting)
# export_chat_history(chat_interface.chatbot)

# ============================================
# CELL 14: Performance Metrics
# ============================================

def show_metrics(chatbot):
    """
    Display chatbot performance metrics
    """
    if not chatbot:
        print("Chatbot not initialized")
        return

    print("📊 Chatbot Performance Metrics")
    print("="*50)
    print(f"Model: {chatbot.model_name}")
    print(f"Temperature: {chatbot.temperature}")
    print(f"Max Tokens: {chatbot.max_tokens}")
    print(f"Vector Store: {'FAISS' if chatbot.use_faiss else 'Chroma'}")
    print(f"Conversations: {len(chatbot.conversation_history)}")

    # Calculate average response length
    if chatbot.conversation_history:
        avg_length = sum(len(conv['assistant']) for conv in chatbot.conversation_history) / len(chatbot.conversation_history)
        print(f"Average Response Length: {avg_length:.0f} characters")

    # Memory usage
    print(f"\nMemory Status:")
    print(f"  - Conversation Memory: {len(chatbot.memory.chat_memory.messages)} messages")
    print(f"  - Vector Store: {'Loaded' if chatbot.vector_store else 'Not loaded'}")

# Show metrics
if 'chat_interface' in locals() and chat_interface.chatbot:
    show_metrics(chat_interface.chatbot)

# ============================================
# CELL 15: Complete Setup Instructions
# ============================================

def show_instructions():
    """Display complete setup instructions"""
    instructions = """
    # 🚀 RAG Chatbot - Google Colab Setup Guide

    ## Quick Start:
    1. **Set API Key**: Add your OpenAI API key in Cell 4
    2. **Initialize**: Click "Initialize Chatbot" button
    3. **Chat**: Start asking questions about your documents

    ## Features:
    - ✅ Document retrieval from uploaded files
    - ✅ Conversation memory across messages
    - ✅ Source attribution for responses
    - ✅ Multiple document formats (TXT, PDF)
    - ✅ Persistent storage with Google Drive

    ## Adding Documents:
    - Run Cell 9 to upload your own documents
    - Supported: .txt and .pdf files
    - Documents are saved to your Google Drive

    ## Customization:
    - Change model: GPT-3.5 or GPT-4
    - Adjust temperature: 0.0 (focused) to 1.0 (creative)
    - Clear history anytime

    ## Tips:
    1. Upload documents before initializing for best results
    2. Be specific in your questions
    3. Chatbot maintains context across messages
    4. Check sources to verify information

    ## Troubleshooting:
    - "Not initialized": Click Initialize button first
    - "API Key error": Check your key in Cell 4
    - "No documents": Upload some or run Cell 5 to create samples

    ## Example Questions:
    - "What is RAG and how does it work?"
    - "Explain the key concepts of AI"
    - "What are the benefits of using LangChain?"
    - "Summarize what we discussed about machine learning"

    Happy chatting! 🎉
    """

    display(Markdown(instructions))

# Show instructions
show_instructions()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142

ModuleNotFoundError: No module named 'langchain.text_splitter'